# 06 — Correlation Analysis: the actual go/no-go

**This notebook answers the research question NCCS/NRC-Cal is built
around:** does NRC-distance (`05`) predict PCE (computed fresh here)?

**PCE formula, already sourced (no new gate needed):** the exact PCE and
Quantile Recalibration formulas were read in full from the QRT paper
(Dheur & Ben Taieb, AISTATS 2024, Section 2) earlier in this project --
implemented in `src/metrics/pce.py`, 12/12 unit tests passing, including
an integration check that QR actually reduces PCE on a deliberately
miscalibrated synthetic model (not just "runs without crashing").

**Two correlations are computed, not one**, because they answer different
questions:
- NRC vs. **PCE(BASE)** — does NRC-distance predict raw miscalibration?
- NRC vs. **PCE(BASE + Quantile Recalibration)** — does NRC-distance
  predict what standard post-hoc recalibration *can't already fix*? This
  is the more valuable question (mirrors how Calibration Bottleneck
  measures residual ECE after Temperature Scaling on the classification
  side) — a diagnostic that only flags what QR already handles isn't
  adding much.

**Decision rule** (agreed earlier in this project):

| \|Spearman rho\| | p-value | Action |
|---|---|---|
| >= 0.5 | < 0.1 | **GO** — proceed to full-scale (`09`) |
| 0.3-0.5 | any | Expand pilot (more datasets) before deciding |
| < 0.3 | any | **NO-GO** on this specific correlation — see `05`'s methodological-tension note (early stopping vs. terminal-phase training) before concluding the whole direction is dead |

**On this run's numbers:** this notebook was built and tested end-to-end
using a **synthetic data fixture** standing in for real UCI downloads (see
`02`'s markdown for why) — the actual correlation values below are
therefore not a real research finding, only a check that the pipeline
computes something sane. The real go/no-go conclusion is only meaningful
once `02` has run against live UCI data.


## Step 1 — Locate project, import helpers

In [ ]:
import sys
from pathlib import Path

if "PROJECT_ROOT" not in dir():
    _here = Path.cwd()
    for candidate in [_here, *_here.parents]:
        if (candidate / "src" / "utils" / "env_utils.py").exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise FileNotFoundError("PROJECT_ROOT not found. Run 00_environment.ipynb first.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.metrics import pce
from src.models import pilot_mixture_model as pmm

print(f"PROJECT_ROOT = {PROJECT_ROOT}")


## Step 2 — Load NRC results (`05`), splits (`02`), and checkpoints (`03`)

In [ ]:
import pickle
import torch
import numpy as np

MIXTURE_SIZE = 1
outputs_dir = PATHS["outputs"] if "PATHS" in dir() else PROJECT_ROOT / "outputs"

EXTERNAL_DIR = PROJECT_ROOT / "external" / "quantile-recalibration-training"
if not EXTERNAL_DIR.exists():
    from src.utils import env_utils
    env_utils.clone_or_pull_repo(
        repo_url="https://github.com/Vekteur/quantile-recalibration-training.git",
        dest=EXTERNAL_DIR, branch="main",
    )
MixturePrediction = pmm.import_mixture_prediction(PROJECT_ROOT)
pmm.set_mixture_prediction_cls(MixturePrediction)

with open(outputs_dir / f"uci_pilot_nrc_mixture_{MIXTURE_SIZE}.pkl", "rb") as f:
    nrc_results = pickle.load(f)["nrc_results"]
with open(outputs_dir / "uci_pilot_splits.pkl", "rb") as f:
    all_splits = pickle.load(f)["splits"]

ckpt_dir = (PATHS["checkpoints"] if "PATHS" in dir() else PROJECT_ROOT / "checkpoints") / f"mixture_{MIXTURE_SIZE}"
print(f"Loaded NRC results and splits for {len(nrc_results)} datasets.")


## Step 3 — Compute PCE(BASE) and PCE(BASE + QR) per dataset

Uses the calibration split to fit the recalibration map (Quantile
Recalibration, `src/metrics/pce.py`) and the held-out test split to
evaluate both PCE numbers -- the calibration split is never touched by
the raw PCE(BASE) evaluation, keeping it a genuinely held-out check.

In [ ]:
import torch.nn.functional as F

def get_mean_std(module, x: np.ndarray) -> tuple:
    x_t = torch.from_numpy(x).to(torch.float32)
    module.model.eval()
    with torch.no_grad():
        means, rhos, _ = module.model(x_t)
        stds = F.softplus(rhos) + 1e-3  # matches MixturePrediction's own std computation
    return means.numpy().ravel(), stds.numpy().ravel()

pce_results = {}
for name, splits in all_splits.items():
    module = pmm.load_pilot_checkpoint(ckpt_dir / f"{name}.pt")

    mean_calib, std_calib = get_mean_std(module, splits["calib"]["x"])
    mean_test, std_test = get_mean_std(module, splits["test"]["x"])

    result = pce.pce_before_and_after_qr(
        mean_calib, std_calib, splits["calib"]["y"],
        mean_test, std_test, splits["test"]["y"],
        M=100,
    )
    pce_results[name] = {"pce_base": result["pce_base"], "pce_qrc": result["pce_qrc"]}
    print(f"  {name:10s}  PCE(BASE)={result['pce_base']:.4f}  PCE(BASE+QR)={result['pce_qrc']:.4f}")


## Step 4 — Merge NRC and PCE into one table

In [ ]:
import pandas as pd

rows = []
for name in nrc_results:
    rows.append({
        "dataset": name,
        "nrc1": nrc_results[name]["nrc1"],
        "nrc2": nrc_results[name]["nrc2"],
        "pce_base": pce_results[name]["pce_base"],
        "pce_qrc": pce_results[name]["pce_qrc"],
        "n_calib_samples": nrc_results[name]["n_calib_samples"],
    })
df = pd.DataFrame(rows).set_index("dataset")
print(df)


## Step 5 — The actual go/no-go: Spearman correlation + decision rule

In [ ]:
from scipy.stats import spearmanr

def evaluate_correlation(x_name, y_name, df):
    rho, p = spearmanr(df[x_name], df[y_name])
    if abs(rho) >= 0.5 and p < 0.1:
        decision = "GO"
    elif abs(rho) >= 0.3:
        decision = "EXPAND PILOT before deciding"
    else:
        decision = "NO-GO on this pair"
    print(f"{x_name} vs {y_name}: rho={rho:.3f}, p={p:.3f}  ->  {decision}")
    return rho, p, decision

print(f"n={len(df)} datasets -- with n=12, treat p-values as indicative, not decisive; "
      f"look at the scatter shape too, not just the threshold table.\n")

results_summary = {}
for x in ("nrc1", "nrc2"):
    for y in ("pce_base", "pce_qrc"):
        results_summary[(x, y)] = evaluate_correlation(x, y, df)


## Step 6 — Save the merged table for `07` (visualization) and `09` (full-scale)

In [ ]:
out_path = outputs_dir / f"uci_pilot_correlation_mixture_{MIXTURE_SIZE}.pkl"
with open(out_path, "wb") as f:
    pickle.dump({"df": df, "correlations": results_summary, "mixture_size": MIXTURE_SIZE}, f)
df.to_csv(outputs_dir / f"uci_pilot_correlation_mixture_{MIXTURE_SIZE}.csv")
print(f"Saved to {out_path} and the accompanying .csv")


## Reading the result

- **If `nrc1`/`nrc2` correlate with `pce_qrc`** (not just `pce_base`): this
  is the strong result — NRC-distance predicts what standard recalibration
  *doesn't already fix*, which is the actual value proposition of NRC-Cal
  as a diagnostic. Proceed toward `08` (closed-form correction) and `09`
  (full-scale, all 57 datasets).
- **If it correlates with `pce_base` but not `pce_qrc`**: still a real
  finding, but a weaker one — NRC-distance is picking up the same signal
  plain post-hoc recalibration already captures, so it adds less on top
  of a standard pipeline. Worth reporting, but the "diagnostic" framing
  weakens; consider whether the closed-form correction (`08`) is still
  differentiated from just running QR.
- **If neither correlates** on real data (unlike this synthetic test run):
  revisit `05`'s methodological-tension note before concluding the
  direction doesn't work — re-run `03` without early stopping on a couple
  of datasets first, since early-stopped training may simply not reach
  the "terminal phase" NRC theory describes.
